# AutoGen Optimization Strategies Notebook

This notebook shows optimization patterns for AutoGen-based agentic workflows.

Topics covered:
- Prefix prompt caching
- Semantic caching (reusing the RAG vector-store directory)
- Prompt compression with LLMLingua
- Concise instruction engineering
- Compact output and max token boundaries
- Task-based model routing
- Batch processing

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import yfinance as yf
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

import autogen

try:
    from llmlingua import PromptCompressor
except Exception:
    PromptCompressor = None

## 0) Configuration

In [ ]:
MODEL_SMALL = 'llama3.1:8b'
MODEL_LARGE = 'llama3.1:70b'
EMBED_MODEL = os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text')
OLLAMA_BASE_URL = os.getenv('OLLAMA_ENDPOINT', 'http://localhost:11434/v1')

PROJECT_ROOT = Path.cwd().parents[2] if 'autogen_financial_agent' in str(Path.cwd()) else Path.cwd()
RAG_VECTOR_DB_DIR = PROJECT_ROOT / 'ai_tutorials' / 'rag' / 'vector_db'
LOCAL_CACHE_DIR = Path.cwd() / 'optimization_cache'
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_BUDGET = 700
MAX_SNIPPETS = 8

print('RAG vector DB directory:', RAG_VECTOR_DB_DIR)

## 1) Prefix caching

In AutoGen loops, keep static system content at the top and change only small suffix fields.

In [ ]:
STATIC_SYSTEM = '''
You are an AutoGen financial analysis agent.
Always follow YAML output format:
ticker:
trend:
top_risks:
confidence:
Only use grounded evidence.
'''

def static_prefix_key(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

print('Static prefix key:', static_prefix_key(STATIC_SYSTEM)[:20], '...')

## 2) Semantic cache with RAG vector-store directory reuse

In [ ]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
semantic_cache = Chroma(
    collection_name='semantic_cache_autogen_finance',
    persist_directory=str(RAG_VECTOR_DB_DIR),
    embedding_function=embeddings,
)

def semantic_put(query: str, answer: str, metadata: dict[str, Any] | None = None) -> None:
    metadata = metadata or {}
    metadata['cached_at'] = datetime.now(timezone.utc).isoformat()
    semantic_cache.add_documents([Document(page_content=query, metadata={**metadata, 'answer': answer})])

def semantic_get(query: str, threshold: float = 0.12):
    hits = semantic_cache.similarity_search_with_score(query, k=1)
    if not hits:
        return None
    doc, dist = hits[0]
    if dist <= threshold:
        return {'query': doc.page_content, 'answer': doc.metadata.get('answer', ''), 'distance': float(dist)}
    return None

semantic_put('Latest financial summary for AAPL', 'Cached example answer for AAPL.', {'ticker': 'AAPL'})
print(semantic_get('Give me latest financial overview of AAPL'))

## 3) Prompt compression with LLMLingua

In [ ]:
long_text = '\n'.join(['Stock valuation depends on growth, margins, and rates.'] * 120)

if PromptCompressor is None:
    print('LLMLingua not installed. Install via: pip install llmlingua')
else:
    compressor = PromptCompressor()
    compressed = compressor.compress_prompt(long_text, rate=0.4)
    print('Original length:', len(long_text))
    print('Compressed length:', len(compressed))

## 4) Concise instructions + compact output format

In [ ]:
CONCISE_SYSTEM = 'Be direct. No pleasantries. No unnecessary explanation. Output YAML only.'

def build_yaml_prompt(ticker: str, question: str, evidence: list[str]) -> str:
    return (
        f'{CONCISE_SYSTEM}\n'
        'Required keys: ticker, trend, top_risks, confidence\n'
        f'Ticker: {ticker}\nQuestion: {question}\n'
        'Evidence:\n' + '\n'.join(f'- {x}' for x in evidence)
    )

print(build_yaml_prompt('MSFT', 'What is current trend?', ['Revenue trend stable', 'Cloud demand strong'])[:300])

## 5) Task-based routing

Choose smaller model by default and escalate only for deeper reasoning tasks.

In [ ]:
def route_model(task: str) -> str:
    t = task.lower()
    if any(k in t for k in ['summary', 'extract', 'classification']):
        return MODEL_SMALL
    if any(k in t for k in ['multi-step reasoning', 'scenario analysis', 'deep analysis']):
        return MODEL_LARGE
    return MODEL_SMALL

print(route_model('summary of latest stock info'))
print(route_model('deep analysis of valuation scenarios'))

## 6) Batch processing for data/tool calls

In [ ]:
def latest_close(ticker: str) -> dict[str, Any]:
    t = yf.Ticker(ticker)
    hist = t.history(period='1d', interval='1d')
    if hist is None or hist.empty:
        return {'ticker': ticker, 'latest_close': None}
    return {'ticker': ticker, 'latest_close': float(hist['Close'].dropna().iloc[-1])}

def batch_latest_close(tickers: list[str]) -> list[dict[str, Any]]:
    return [latest_close(t) for t in tickers]

print(json.dumps(batch_latest_close(['NVDA', 'AAPL', 'MSFT']), indent=2, ensure_ascii=True))

## 7) AutoGen optimized orchestration sketch

This combines: semantic cache lookup, task routing, and bounded generation settings.

In [ ]:
def autogen_optimized_answer(ticker: str, question: str) -> dict[str, Any]:
    cached = semantic_get(question, threshold=0.12)
    if cached:
        return {'path': 'semantic_cache', 'payload': cached}

    model = route_model('summary')

    llm_config = {
        'config_list': [{'model': model, 'base_url': OLLAMA_BASE_URL, 'api_key': 'ollama'}],
        'temperature': 0,
        'timeout': 90,
        'max_tokens': 180,
    }

    planner = autogen.AssistantAgent(
        name='planner',
        llm_config=llm_config,
        system_message='Call tool first for stock evidence, then reply compactly.',
    )

    executor = autogen.UserProxyAgent(
        name='exec',
        human_input_mode='NEVER',
        code_execution_config=False,
    )

    autogen.register_function(
        latest_close,
        caller=planner,
        executor=executor,
        name='latest_close',
        description='Get latest close for ticker.',
    )

    result = executor.initiate_chat(
        planner,
        message=f'Ticker: {ticker}. Question: {question}. Use latest_close then answer.',
        max_turns=4,
    )

    final_msg = (result.chat_history[-1].get('content') if result.chat_history else '')
    semantic_put(question, final_msg, metadata={'ticker': ticker, 'model': model})
    return {'path': 'llm', 'model': model, 'answer': final_msg}

print(autogen_optimized_answer('NVDA', 'Give short latest financial trend.'))